In [8]:
import scanpy as sc
import numpy as np
import pandas as pd
import scipy.sparse as sp

# ── 重新讀GEX和preprocessing ──────────────────────────
adata_gex = sc.read_10x_h5(
    '/Users/renegibson/Downloads/Breast_Cancer_3p_filtered_feature_bc_matrix.h5',
    gex_only=True
)
adata_gex.var_names_make_unique()

adata_gex.var['mt'] = adata_gex.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata_gex, qc_vars=['mt'], inplace=True)
sc.pp.filter_cells(adata_gex, min_genes=200)
sc.pp.filter_genes(adata_gex, min_cells=3)
adata_gex = adata_gex[
    (adata_gex.obs['pct_counts_mt'] > 1) &
    (adata_gex.obs['pct_counts_mt'] < 20) &
    (adata_gex.obs['total_counts'] > 2000)
].copy()

import scrublet as scr
scrub = scr.Scrublet(adata_gex.X)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata_gex.obs['predicted_doublet'] = predicted_doublets
adata_gex = adata_gex[~adata_gex.obs['predicted_doublet']].copy()

sc.pp.normalize_total(adata_gex, target_sum=1e4)
sc.pp.log1p(adata_gex)
adata_gex.raw = adata_gex
sc.pp.highly_variable_genes(adata_gex, n_top_genes=2000)
sc.pp.pca(adata_gex)
sc.pp.neighbors(adata_gex)
sc.tl.umap(adata_gex)
sc.tl.leiden(adata_gex, resolution=1.0,
             flavor='igraph', directed=False, n_iterations=2,
             key_added='leiden_high')

# Cell type annotation
cell_type_map = {
    '0':  'Tumor_Luminal',
    '1':  'Tumor_heatshock',
    '2':  'Tumor_Luminal',
    '3':  'Tumor_Luminal',
    '4':  'Macrophage',
    '5':  'Tumor_Neuroendocrine',
    '6':  'Endothelial',
    '7':  'Tumor_Luminal',
    '8':  'Tumor_DNAstress',
    '9':  'Tumor_RGS5high',
    '10': 'Tumor_Proliferating',
    '11': 'Tumor_LuminalA',
    '12': 'Tumor_RGS5high',
    '13': 'Fibroblast',
    '14': 'Macrophage',
    '15': 'Dying',
    '16': 'Smooth_muscle',
}
adata_gex.obs['cell_type'] = adata_gex.obs['leiden_high'].map(cell_type_map)
adata_gex = adata_gex[adata_gex.obs['cell_type'] != 'Dying'].copy()

# Fix barcode format
adata_gex.obs_names = pd.Index(
    ['IDC_' + bc.replace('-1', '') for bc in adata_gex.obs_names]
)

# ── Load IsoCAPE and IsoDecipher ──────────────────────
cape = sc.read_h5ad('/Users/renegibson/Downloads/IDC_cape.h5ad')
apa  = sc.read_h5ad(
    '/Users/renegibson/Desktop/githubrepo/IsoDecipher/results/master_mosaic_combined_IDC.h5ad'
)

common_cape = adata_gex.obs_names.intersection(cape.obs_names)
common_apa  = adata_gex.obs_names.intersection(apa.obs_names)
print(f"Common with IsoCAPE:     {len(common_cape)}")
print(f"Common with IsoDecipher: {len(common_apa)}")

# ── IsoCAPE layer ─────────────────────────────────────
cell_to_idx = {c: i for i, c in enumerate(adata_gex.obs_names)}
n_cape = cape.n_vars
cape_mat = sp.lil_matrix((adata_gex.n_obs, n_cape), dtype=np.float32)
for cell in common_cape:
    cape_mat[cell_to_idx[cell]] = cape[cell].X
adata_gex.obsm['isocape'] = sp.csr_matrix(cape_mat)
adata_gex.uns['isocape_features'] = cape.var_names.tolist()
adata_gex.uns['isocape_var']      = cape.var.to_dict()

# ── IsoDecipher layer ─────────────────────────────────
iso_features = apa.var_names[apa.var['feature_types']=='Isoform']
n_iso = len(iso_features)
iso_mat = sp.lil_matrix((adata_gex.n_obs, n_iso), dtype=np.float32)
for cell in common_apa:
    iso_mat[cell_to_idx[cell]] = apa[cell, iso_features].X
adata_gex.obsm['isoform'] = sp.csr_matrix(iso_mat)
adata_gex.uns['isoform_features'] = iso_features.tolist()

cape_feat = cape.var_names.tolist()
iso_feat  = iso_features.tolist()

print(f"\nCells: {adata_gex.n_obs}")
print(f"obsm['isocape']:  {n_cape:,} sites")
print(f"obsm['isoform']:  {n_iso:,} features")
print(adata_gex.obs['cell_type'].value_counts())

# ── Save ──────────────────────────────────────────────
adata_gex.write_h5ad('/Users/renegibson/Downloads/IDC_gex_annotated.h5ad')
print("\nSaved → IDC_gex_annotated.h5ad")

/Users/renegibson/miniforge3/envs/iso_decipher/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
/Users/renegibson/miniforge3/envs/iso_decipher/lib/python3.10/site-packages/anndata/_core/anndata.py:1758: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
OMP: Info #276: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.64
Detected doublet rate = 0.2%
Estimated detectable doublet fraction = 7.2%
Overall doublet rate:
	Expected   = 10.0%
	Estimated  = 2.8%
Elapsed time: 2.5 seconds
Common with IsoCAPE:     2974
Common with IsoDecipher: 2974

Cells: 2974
obsm['isocape']:  153,156 sites
obsm['isoform']:  63,825 features
cell_type
Tumor_Luminal           948
Tumor_DNAstress         519
Tumor_RGS5high          491
Tumor_heatshock         437
Macrophage              201
Tumor_Neuroendocrine    139
Tumor_Proliferating      59
Tumor_LuminalA           54
Fibroblast               51
Endothelial              46
Smooth_muscle            29
Name: count, dtype: int64

Saved → IDC_gex_annotated.h5ad
